In [1]:
!pip install transformers datasets peft accelerate bitsandbytes
!pip install rouge-score nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=eabd97668d465295ea09299096ab9ad54ed4ad47bfda3e4c0e3f0babb736f9d9
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [2]:
import os
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, pipeline
from peft import LoraConfig, get_peft_model
import torch

In [3]:

# 1. LOAD DATASET (Assignment 1)
dataset = load_dataset("ccdv/arxiv-summarization", split="train[:1500]")

def format_data(example):
    return {
        "text": f"### Instruction: Summarize the following research:\n{example['article'][:300]}\n### Response: {example['abstract']}"
    }

dataset = dataset.map(format_data)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

section/train-00000-of-00015.parquet:   0%|          | 0.00/230M [00:00<?, ?B/s]

section/train-00001-of-00015.parquet:   0%|          | 0.00/228M [00:00<?, ?B/s]

section/train-00002-of-00015.parquet:   0%|          | 0.00/228M [00:00<?, ?B/s]

...

section/test-00000-of-00001.parquet:   0%|          | 0.00/105M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/203037 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6436 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6440 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [4]:
# 2. LOAD BASE MODEL
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

base_model.config.use_cache = False

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
# 3. APPLY LoRA (Fine-tuning)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

model = get_peft_model(base_model, lora_config)


In [6]:
# 4. TOKENIZATION
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize)

tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [7]:
# 5. TRAINING
training_args = TrainingArguments(
    output_dir="./finetuned_model",
    per_device_train_batch_size=1,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=50,
    fp16=True,
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()


{'loss': '4.101', 'grad_norm': '1.084', 'learning_rate': '4.99e-05', 'epoch': '0.006667'}
{'loss': '3.181', 'grad_norm': '8.342', 'learning_rate': '4.979e-05', 'epoch': '0.01333'}
{'loss': '4.048', 'grad_norm': '1.361', 'learning_rate': '4.968e-05', 'epoch': '0.02'}
{'loss': '3.467', 'grad_norm': '1.496', 'learning_rate': '4.957e-05', 'epoch': '0.02667'}
{'loss': '3.384', 'grad_norm': '1.562', 'learning_rate': '4.946e-05', 'epoch': '0.03333'}
{'loss': '3.882', 'grad_norm': '2.259', 'learning_rate': '4.934e-05', 'epoch': '0.04'}
{'loss': '3.002', 'grad_norm': '2.036', 'learning_rate': '4.923e-05', 'epoch': '0.04667'}
{'loss': '3.015', 'grad_norm': '1.86', 'learning_rate': '4.912e-05', 'epoch': '0.05333'}
{'loss': '2.871', 'grad_norm': '1.806', 'learning_rate': '4.901e-05', 'epoch': '0.06'}
{'loss': '2.761', 'grad_norm': '2.379', 'learning_rate': '4.89e-05', 'epoch': '0.06667'}
{'loss': '2.801', 'grad_norm': '21.55', 'learning_rate': '4.879e-05', 'epoch': '0.07333'}
{'loss': '2.565', 'gr

TrainOutput(global_step=4500, training_loss=2.124589799669054, metrics={'train_runtime': 641.4024, 'train_samples_per_second': 7.016, 'train_steps_per_second': 7.016, 'train_loss': 2.124589799669054, 'epoch': 3.0})

In [8]:
# 6. SAVE MODEL
model.save_pretrained("./finetuned_model")
tokenizer.save_pretrained("./finetuned_model")


('./finetuned_model/tokenizer_config.json',
 './finetuned_model/chat_template.jinja',
 './finetuned_model/tokenizer.json')

In [9]:
# 7. LOAD PIPELINES
base_pipe = pipeline(
    "text-generation",
    model=model_name,
    tokenizer=model_name
)

finetuned_pipe = pipeline(
    "text-generation",
    model="./finetuned_model",
    tokenizer="./finetuned_model"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [10]:
# 8. COMPARISON
prompt = "### Instruction: Explain gradient descent\n### Response:"

base_output = base_pipe(prompt, max_new_tokens=100)[0]["generated_text"]
finetuned_output = finetuned_pipe(prompt, max_new_tokens=100)[0]["generated_text"]

print("\n===== BASE MODEL OUTPUT =====\n")
print(base_output)

print("\n===== FINE-TUNED MODEL OUTPUT =====\n")
print(finetuned_output)



===== BASE MODEL OUTPUT =====

### Instruction: Explain gradient descent
### Response: Gradient descent is an iterative optimization algorithm that learns a solution to a given problem by iteratively approximating the solution using a sequence of small steps. It is a popular optimization method for solving nonlinear optimization problems, such as minimizing the sum of squares of the errors in a neural network. The algorithm first initializes the parameters of the network with random values, then updates these parameters based on the gradient of the loss function with respect to each parameter. The algorithm continues to update the parameters

===== FINE-TUNED MODEL OUTPUT =====

### Instruction: Explain gradient descent
### Response: we demonstrate the use of gradient descent on a small neural network to learn a sparse representation of a large dataset. we show that gradient descent can efficiently learn the sparse representation for a fully connected network on a synthetic dataset an

In [11]:
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk

nltk.download('punkt')

# Reference (expected output from dataset)
example = dataset[0]
prompt = f"### Instruction: Explain the following:\n{example['article'][:1000]}\n### Response:"
reference = example["abstract"]
base_output = base_pipe(prompt, max_new_tokens=100)[0]["generated_text"]
finetuned_output = finetuned_pipe(prompt, max_new_tokens=100)[0]["generated_text"]

# Extract generated responses
base_resp = base_output.split("### Response:")[-1].strip()
fine_resp = finetuned_output.split("### Response:")[-1].strip()

# ROUGE
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

base_rouge = scorer.score(reference, base_resp)
fine_rouge = scorer.score(reference, fine_resp)

# BLEU
smooth = SmoothingFunction().method1

base_bleu = sentence_bleu([reference.split()], base_resp.split(), smoothing_function=smooth)
fine_bleu = sentence_bleu([reference.split()], fine_resp.split(), smoothing_function=smooth)

print("\n===== METRICS COMPARISON =====\n")

print("Base Model ROUGE-1:", base_rouge['rouge1'].fmeasure)
print("Fine-tuned Model ROUGE-1:", fine_rouge['rouge1'].fmeasure)

print("Base Model ROUGE-L:", base_rouge['rougeL'].fmeasure)
print("Fine-tuned Model ROUGE-L:", fine_rouge['rougeL'].fmeasure)

print("Base Model BLEU:", base_bleu)
print("Fine-tuned Model BLEU:", fine_bleu)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.



===== METRICS COMPARISON =====

Base Model ROUGE-1: 0.29126213592233013
Fine-tuned Model ROUGE-1: 0.07246376811594203
Base Model ROUGE-L: 0.15533980582524273
Fine-tuned Model ROUGE-L: 0.07246376811594203
Base Model BLEU: 0.005364322042722357
Fine-tuned Model BLEU: 1.7455202174842014e-08


In [12]:
# 9. INTERACTIVE CHATBOT
print("\nModel ready! Type 'exit' to stop.\n")

while True:
    user_input = input("Enter your question: ")

    if user_input.lower() == "exit":
        break

    prompt = f"### Instruction: Explain in detail: {user_input}\n### Response:"

    output = finetuned_pipe(
        prompt,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    answer = output[0]["generated_text"].split("### Response:")[-1].strip()

    print("\nAnswer:", answer)
    print("-" * 50)


Model ready! Type 'exit' to stop.

Enter your question: explain gradient descent

Answer: we introduce a new algorithm for solving linear systems using the gradient descent method. 
 we show that the gradient descent method is equivalent to a variant of the primal-dual algorithm for solving a linear system of the form @xmath0. 
 we also show that the gradient descent method is optimal in the sense that the primal-dual algorithm is not better than the gradient descent method in terms of convergence rate. 
 we also show that the gradient descent method is not better than the primal-dual algorithm in terms of computational complexity. 
 in particular, the gradient descent method is slower than the primal-dual algorithm in terms of computational complexity. 
 the algorithm is simple and can be implemented in
--------------------------------------------------
Enter your question: what is overfitting

Answer: we present a novel approach for training a neural network in the context of superv